In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


Loop through folder + visualize (QA mode)

Batch processing + save (Production mode)

In [ ]:
from pathlib import Path
import cv2
import numpy as np

INPUT_DIR = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/images")
OUT_GRAY = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_clahe")
OUT_COLOR = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/color_clahe")

OUT_GRAY.mkdir(parents=True, exist_ok=True)
OUT_COLOR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}

clahe_gray = cv2.createCLAHE(clipLimit=4.5, tileGridSize=(8, 8))
clahe_color = cv2.createCLAHE(clipLimit=4.5, tileGridSize=(8, 8))

image_paths = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS]

for img_path in image_paths:
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue

    # ---- Grayscale ----
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    gray = clahe_gray.apply(gray.astype(np.uint8))

    cv2.imwrite(str(OUT_GRAY / img_path.name), gray)

    # ---- Color (LAB + CLAHE on L) ----
    bgr = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = clahe_color.apply(l.astype(np.uint8))
    lab = cv2.merge((l, a, b))
    color = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    cv2.imwrite(str(OUT_COLOR / img_path.name), color)

print("Batch preprocessing complete")


RESIZING (Without DATA FLIPPING, etc)

In [ ]:
from pathlib import Path
import cv2
import numpy as np

# =========================
# Config
# =========================
INPUT_DIR = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/images")
OUT_GRAY = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_clahe_512_noborder")
OUT_COLOR = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/color_clahe_512_noborder")

OUT_GRAY.mkdir(parents=True, exist_ok=True)
OUT_COLOR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
TARGET_SIZE = (1500, 1000)  # (width, height)

clahe_gray = cv2.createCLAHE(clipLimit=4.5, tileGridSize=(8, 8))
clahe_color = cv2.createCLAHE(clipLimit=4.5, tileGridSize=(8, 8))

# =========================
# OPTIONAL: Trim existing black borders first
# (useful if your raw images already have black bands)
# =========================
def trim_black_borders(img, thresh=8):
    """
    Removes near-black borders by finding the bounding box of non-black pixels.
    Works for grayscale and BGR.
    """
    if img.ndim == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Non-black mask
    mask = gray > thresh
    coords = np.argwhere(mask)

    if coords.size == 0:
        return img  # image is basically all black; give up

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1  # +1 for slicing
    return img[y0:y1, x0:x1]

# =========================
# Resize without borders: crop-to-fill
# =========================
def resize_crop_fill(img, target_size=(1500, 1000)):
    """
    Scale so the image fully covers target size, then center-crop.
    No padding -> no borders.
    """
    h, w = img.shape[:2]
    target_w, target_h = target_size

    # Scale UP so both dimensions >= target
    scale = max(target_w / w, target_h / h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))

    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Center crop to target
    x0 = (new_w - target_w) // 2
    y0 = (new_h - target_h) // 2

    cropped = resized[y0:y0 + target_h, x0:x0 + target_w]
    return cropped

# =========================
# Batch processing
# =========================
image_paths = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS]
print(f"Found {len(image_paths)} images")

for img_path in image_paths:
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue

    # Optional: trim black borders in raw
    img_bgr = trim_black_borders(img_bgr, thresh=8)

    # ---------- Grayscale pipeline ----------
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    gray = clahe_gray.apply(gray.astype(np.uint8))
    gray = resize_crop_fill(gray, TARGET_SIZE)

    cv2.imwrite(str(OUT_GRAY / img_path.name), gray)

    # ---------- Color pipeline (LAB + CLAHE on L) ----------
    bgr = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = clahe_color.apply(l.astype(np.uint8))
    lab = cv2.merge((l, a, b))
    color = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    color = resize_crop_fill(color, TARGET_SIZE)

    cv2.imwrite(str(OUT_COLOR / img_path.name), color)

print("Batch preprocessing + border-free resize complete")


Data flipping and augmentation for more data (Balancing of labels)

In [ ]:
from pathlib import Path
import json
import cv2
import numpy as np
import random
from collections import Counter, defaultdict

# =========================
# Config
# =========================
INPUT_DIR = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/images")
JSON_PATH = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/result_labels.json")

OUT_GRAY = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_512_noborder_balanced")
OUT_COLOR = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/color_512_noborder_balanced")

OUT_GRAY.mkdir(parents=True, exist_ok=True)
OUT_COLOR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
TARGET_SIZE = (1500, 1000)

random.seed(42)
np.random.seed(42)

TARGET_LABELS = {"overlap", "isolated", "contraband", "non_contraband"}


# =========================
# Label Studio result.json parser
# =========================
def normalize_label(label):
    label = label.lower().strip()
    label = label.replace("-", "_").replace(" ", "_")

    if label in ["overlap", "overlapping"]:
        return "overlap"
    if label in ["isolated", "isolate", "no_overlap", "non_overlap"]:
        return "isolated"
    if label in ["contraband", "threat"]:
        return "contraband"
    if label in ["non_contraband", "noncontraband", "non_threat", "safe"]:
        return "non_contraband"

    return label


def extract_filename_from_task(task):
    data = task.get("data", {})

    for key in ["image", "img", "file", "filename"]:
        if key in data:
            return Path(str(data[key])).name

    return None


def extract_labels_from_task(task):
    labels = set()

    for ann in task.get("annotations", []) + task.get("predictions", []):
        for r in ann.get("result", []):
            value = r.get("value", {})

            for key in ["polygonlabels", "rectanglelabels", "brushlabels", "labels"]:
                if key in value:
                    for label in value[key]:
                        labels.add(normalize_label(label))

            if "choices" in value:
                for choice in value["choices"]:
                    labels.add(normalize_label(choice))

    return labels & TARGET_LABELS


def load_labels_from_result_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        tasks = json.load(f)

    image_to_labels = {}

    for task in tasks:
        filename = extract_filename_from_task(task)
        if filename is None:
            continue

        labels = extract_labels_from_task(task)
        image_to_labels[filename] = labels

    return image_to_labels


# =========================
# Image preprocessing
# =========================
def trim_black_borders(img, thresh=8):
    if img.ndim == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    mask = gray > thresh
    coords = np.argwhere(mask)

    if coords.size == 0:
        return img

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1

    return img[y0:y1, x0:x1]


def resize_crop_fill(img, target_size=(1500, 1000)):
    h, w = img.shape[:2]
    target_w, target_h = target_size

    scale = max(target_w / w, target_h / h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))

    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    x0 = (new_w - target_w) // 2
    y0 = (new_h - target_h) // 2

    return resized[y0:y0 + target_h, x0:x0 + target_w]


def preprocess_gray(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    gray = resize_crop_fill(gray, TARGET_SIZE)
    return gray


def preprocess_color(img_bgr):
    color = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    color = resize_crop_fill(color, TARGET_SIZE)
    return color


# =========================
# Augmentation for minority labels
# =========================
def augment_flip_only(img):
    return cv2.flip(img, 1)


# =========================
# Load labels
# =========================
image_to_labels = load_labels_from_result_json(JSON_PATH)

image_paths = sorted([
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTS
])

valid_items = []
label_to_paths = defaultdict(list)

for p in image_paths:
    labels = image_to_labels.get(p.name, set())

    if len(labels) == 0:
        print(f"WARNING: No usable label found for {p.name}")
        continue

    valid_items.append((p, labels))

    for label in labels:
        label_to_paths[label].append(p)

print("Original label counts from result.json:")
for label in ["overlap", "isolated", "contraband", "non_contraband"]:
    print(f"{label}: {len(label_to_paths[label])}")


# =========================
# Balance labels by flipping minority images
# =========================
target_count = max(len(paths) for paths in label_to_paths.values())
print(f"\nBalancing target count: {target_count}")

current_counts = Counter()

for _, labels in valid_items:
    for label in labels:
        current_counts[label] += 1

extra_aug_per_path = Counter()

while True:
    minority_labels = [
        label for label in ["overlap", "isolated", "contraband", "non_contraband"]
        if current_counts[label] < target_count
    ]

    if not minority_labels:
        break

    label = min(minority_labels, key=lambda x: current_counts[x])
    candidates = label_to_paths[label]

    chosen_path = random.choice(candidates)
    extra_aug_per_path[chosen_path] += 1

    chosen_labels = image_to_labels[chosen_path.name] & TARGET_LABELS

    for l in chosen_labels:
        current_counts[l] += 1


print("\nPlanned final label counts:")
for label in ["overlap", "isolated", "contraband", "non_contraband"]:
    print(f"{label}: {current_counts[label]}")

print(f"\nExtra flipped images to create: {sum(extra_aug_per_path.values())}")

UPDATED_JSON_PATH = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/result_labels_balanced.json")

balanced_label_records = []


# =========================
# Process and save + update label JSON
# =========================
final_counts = Counter()

for img_path, labels in valid_items:
    img_bgr = cv2.imread(str(img_path))

    if img_bgr is None:
        print(f"WARNING: Could not read {img_path}")
        continue

    img_bgr = trim_black_borders(img_bgr, thresh=8)

    gray = preprocess_gray(img_bgr)
    color = preprocess_color(img_bgr)

    base_name = img_path.stem

    # ---------- Save original processed image ----------
    gray_orig_name = f"{base_name}_orig.png"
    color_orig_name = f"{base_name}_orig.png"

    cv2.imwrite(str(OUT_GRAY / gray_orig_name), gray)
    cv2.imwrite(str(OUT_COLOR / color_orig_name), color)

    # Add label entry for original processed image
    balanced_label_records.append({
        "image": gray_orig_name,
        "gray_path": str(OUT_GRAY / gray_orig_name),
        "color_path": str(OUT_COLOR / color_orig_name),
        "source_image": img_path.name,
        "augmentation": "original",
        "labels": sorted(list(labels)),
    })

    for label in labels:
        final_counts[label] += 1

    # ---------- Save flipped minority augmentations ----------
    n_extra = extra_aug_per_path[img_path]

    for i in range(n_extra):
        gray_aug = augment_flip_only(gray)
        color_aug = augment_flip_only(color)

        gray_aug_name = f"{base_name}_balflip{i:03d}.png"
        color_aug_name = f"{base_name}_balflip{i:03d}.png"

        cv2.imwrite(str(OUT_GRAY / gray_aug_name), gray_aug)
        cv2.imwrite(str(OUT_COLOR / color_aug_name), color_aug)

        # Add label entry for flipped image
        balanced_label_records.append({
            "image": gray_aug_name,
            "gray_path": str(OUT_GRAY / gray_aug_name),
            "color_path": str(OUT_COLOR / color_aug_name),
            "source_image": img_path.name,
            "augmentation": "horizontal_flip",
            "labels": sorted(list(labels)),
        })

        for label in labels:
            final_counts[label] += 1


# =========================
# Save updated balanced label JSON
# =========================
with open(UPDATED_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(balanced_label_records, f, indent=2)

print("\nFinal saved label counts:")
for label in ["overlap", "isolated", "contraband", "non_contraband"]:
    print(f"{label}: {final_counts[label]}")

print(f"\nSaved updated balanced label JSON to:")
print(UPDATED_JSON_PATH)

print("\nDone. No CLAHE used. Original data kept. Minority labels balanced using horizontal flipping.")

Without clahe


In [ ]:
from pathlib import Path
import cv2
import numpy as np

# =========================
# Config
# =========================
INPUT_DIR = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/images")
OUT_GRAY = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_512_noborder")
OUT_COLOR = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/color_512_noborder")

OUT_GRAY.mkdir(parents=True, exist_ok=True)
OUT_COLOR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
TARGET_SIZE = (1500, 1000)  # (width, height)

# =========================
# Trim existing black borders first
# =========================
def trim_black_borders(img, thresh=8):
    if img.ndim == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    mask = gray > thresh
    coords = np.argwhere(mask)

    if coords.size == 0:
        return img

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1

    return img[y0:y1, x0:x1]

# =========================
# Resize without borders: crop-to-fill
# =========================
def resize_crop_fill(img, target_size=(1500, 1000)):
    h, w = img.shape[:2]
    target_w, target_h = target_size

    scale = max(target_w / w, target_h / h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))

    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    x0 = (new_w - target_w) // 2
    y0 = (new_h - target_h) // 2

    return resized[y0:y0 + target_h, x0:x0 + target_w]

# =========================
# Batch processing
# =========================
image_paths = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS]
print(f"Found {len(image_paths)} images")

for img_path in image_paths:
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue

    img_bgr = trim_black_borders(img_bgr, thresh=8)

    # ---------- Grayscale pipeline ----------
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    gray = resize_crop_fill(gray, TARGET_SIZE)

    cv2.imwrite(str(OUT_GRAY / img_path.name), gray)

    # ---------- Color pipeline ----------
    color = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    color = resize_crop_fill(color, TARGET_SIZE)

    cv2.imwrite(str(OUT_COLOR / img_path.name), color)

print("Batch preprocessing without CLAHE complete")